In [43]:
# ===============================================================
# Lahore Buildings (Google Open Buildings Temporal v1)
# Per-pixel presence/height, high-rise mask, point grid, UC stats
# ===============================================================
import os, ee, geemap, geopandas as gpd, pandas as pd

# ---------- 0) EE init ----------
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

# ---------- 1) Inputs ----------
UC_SHP = "../../data/Lahore UCs/Lahore UC.shp"   # your UC polygons
YEAR   = 2023                                    # any year 2016..2023
PRESENCE_THRESH = 0.5                             # presence threshold (uncalibrated; tune 0.4–0.6)
HIGHRISE_HEIGHT = 18.0                            # meters; ~6 floors ≈ 3 m/floor
POINT_SAMPLE_M  = 50                              # point grid spacing (20–50 m is reasonable)
OUT_PREFIX = f"OpenBuildings_Lahore_{YEAR}"




In [44]:

gdf = gpd.read_file(UC_SHP).to_crs(4326)
gdf["geometry"] = gdf["geometry"].buffer(0)
ucs_fc = geemap.gdf_to_ee(gdf)
region = ucs_fc.geometry()
id_field = "UC" if "UC" in gdf.columns else gdf.columns[0]

# ----- dataset -----
COL = "GOOGLE/Research/open-buildings-temporal/v1"
col = (ee.ImageCollection(COL)
       .filterBounds(region)
       .filterDate(f"{YEAR}-01-01", f"{YEAR}-12-31"))
mosaic = col.mosaic()
presence = mosaic.select("building_presence")
height = mosaic.select("building_height")
built_mask = presence.gte(PRESENCE_THRESH)
highrise_mask = built_mask.And(height.gte(HIGHRISE_HEIGHT))

# ----- sample points (~POINT_SAMPLE_M grid) -----
pts_img = (ee.Image.pixelLonLat()
           .addBands(presence.rename("presence"))
           .addBands(height.rename("height"))
           .addBands(highrise_mask.rename("highrise")))
pts_fc = pts_img.sample(region=region, scale=POINT_SAMPLE_M,
                        geometries=True, seed=1, tileScale=4)
# assumes you already created: region, presence, height, highrise_mask, ucs_fc
# If not: re-use the earlier cell that built `mosaic`, `presence`, `height`, etc.


# Build the points FeatureCollection (server-side)
pts_img = (ee.Image.pixelLonLat()
           .addBands(presence.rename("presence"))
           .addBands(height.rename("height"))
           .addBands(highrise_mask.int().rename("highrise")))   # booleans → int

pts_fc = pts_img.sample(
    region=region,
    scale=POINT_SAMPLE_M,
    geometries=True,
    seed=1
)

# ---- Export to Google Drive: GeoJSON (standalone, big collections OK)
task_gj = ee.batch.Export.table.toDrive(
    collection=pts_fc.select(['longitude','latitude','presence','height','highrise']),
    description=f'{OUT_PREFIX}_geojson',
    folder='EE_Exports',                 # change if you like
    fileNamePrefix=OUT_PREFIX,
    fileFormat='GeoJSON'
)
task_gj.start()

# ---- Export to Google Drive: CSV (fast for Folium HeatMap)
task_csv = ee.batch.Export.table.toDrive(
    collection=pts_fc.select(['longitude','latitude','presence','height','highrise']),
    description=f'{OUT_PREFIX}_csv',
    folder='EE_Exports',
    fileNamePrefix=f'{OUT_PREFIX}',
    fileFormat='CSV'
)
task_csv.start()

print("Started Drive exports:\n - GeoJSON\n - CSV\nCheck https://code.earthengine.google.com/tasks for progress.")


Started Drive exports:
 - GeoJSON
 - CSV
Check https://code.earthengine.google.com/tasks for progress.


In [45]:
import geopandas as gpd
gdf_pts = gpd.read_file("OpenBuildings_Lahore_2023.geojson")
if gdf_pts.crs is None or gdf_pts.crs.to_epsg() != 4326:
    gdf_pts = gdf_pts.set_crs(4326, allow_override=True)
shp_dir = f"exports/{OUT_PREFIX}_{POINT_SAMPLE_M}m_shp"
os.makedirs(shp_dir, exist_ok=True)
gdf_pts.to_file(os.path.join(shp_dir, f"{OUT_PREFIX}_{POINT_SAMPLE_M}m.shp"),
                driver="ESRI Shapefile")
print("[OK] Shapefile set →", shp_dir)

[OK] Shapefile set → exports/OpenBuildings_Lahore_2023_50m_shp


In [33]:

# ----- region -----
gdf = gpd.read_file(UC_SHP).to_crs(4326)
gdf["geometry"] = gdf["geometry"].buffer(0)
ucs_fc = geemap.gdf_to_ee(gdf)
region = ucs_fc.geometry()
id_field = "UC" if "UC" in gdf.columns else gdf.columns[0]

# ----- dataset -----
COL = "GOOGLE/Research/open-buildings-temporal/v1"
col = (ee.ImageCollection(COL)
       .filterBounds(region)
       .filterDate(f"{YEAR}-01-01", f"{YEAR}-12-31"))
mosaic = col.mosaic()
presence = mosaic.select("building_presence")
height = mosaic.select("building_height")
built_mask = presence.gte(PRESENCE_THRESH)
highrise_mask = built_mask.And(height.gte(HIGHRISE_HEIGHT))

# ----- sample points (~POINT_SAMPLE_M grid) -----
pts_img = (ee.Image.pixelLonLat()
           .addBands(presence.rename("presence"))
           .addBands(height.rename("height"))
           .addBands(highrise_mask.rename("highrise")))
pts_fc = pts_img.sample(region=region, scale=POINT_SAMPLE_M,
                        geometries=True, seed=1, tileScale=4)

# ----- export vector -----
os.makedirs("exports", exist_ok=True)
geojson_pts = f"exports/{OUT_PREFIX}_{POINT_SAMPLE_M}m.geojson"
geemap.ee_export_vector(pts_fc, filename=geojson_pts)
print("[OK] GeoJSON points →", geojson_pts)

# ----- also CSV & shapefile -----
try:
    dfp = geemap.ee_to_df(pts_fc)
except Exception:
    info = ee.FeatureCollection(pts_fc).getInfo()
    dfp = pd.DataFrame([f["properties"] for f in info["features"]])
dfp = dfp.rename(columns={"latitude":"lat","longitude":"lon"})
dfp[["lat","lon","presence","height","highrise"]].to_csv(
    f"exports/{OUT_PREFIX}_{POINT_SAMPLE_M}m.csv", index=False)
print("[OK] CSV points →", f"exports/{OUT_PREFIX}_{POINT_SAMPLE_M}m.csv")

import geopandas as gpd
gdf_pts = gpd.read_file(geojson_pts)
if gdf_pts.crs is None or gdf_pts.crs.to_epsg() != 4326:
    gdf_pts = gdf_pts.set_crs(4326, allow_override=True)
shp_dir = f"exports/{OUT_PREFIX}_{POINT_SAMPLE_M}m_shp"
os.makedirs(shp_dir, exist_ok=True)
gdf_pts.to_file(os.path.join(shp_dir, f"{OUT_PREFIX}_{POINT_SAMPLE_M}m.shp"),
                driver="ESRI Shapefile")
print("[OK] Shapefile set →", shp_dir)




Generating URL ...
Please wait ...
An error occurred while downloading. 
 Retrying ...
Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/Building+Highrise/exports/OpenBuildings_Lahore_2023_50m.geojson
[OK] GeoJSON points → exports/OpenBuildings_Lahore_2023_50m.geojson


KeyboardInterrupt: 

In [ ]:
# ----- UC-wise statistics -----
pixel_area = ee.Image.pixelArea()
built_area = pixel_area.updateMask(built_mask)
highrise_area = pixel_area.updateMask(highrise_mask)

reducers = ee.Reducer.mean().combine(
    reducer2=ee.Reducer.median(), sharedInputs=True
).combine(
    reducer2=ee.Reducer.minMax(), sharedInputs=True
)

stats_presence = presence.reduceRegions(ucs_fc, ee.Reducer.mean(), scale=4, tileScale=2)
stats_height = height.updateMask(built_mask).reduceRegions(ucs_fc, reducers, scale=4, tileScale=2)
stats_built = built_area.reduceRegions(ucs_fc, ee.Reducer.sum(), scale=4, tileScale=2)
stats_high = highrise_area.reduceRegions(ucs_fc, ee.Reducer.sum(), scale=4, tileScale=2)

def _fc_to_df(fc):
    try:
        return geemap.ee_to_df(fc)
    except Exception:
        info = ee.FeatureCollection(fc).getInfo()
        return pd.DataFrame([f["properties"] for f in info["features"]])

dfp_ = _fc_to_df(stats_presence).rename(columns={"mean":"presence_mean"})
dfh_ = _fc_to_df(stats_height).rename(columns={
    "mean":"height_mean","median":"height_median","min":"height_min","max":"height_max"})
dfb_ = _fc_to_df(stats_built).rename(columns={"sum":"built_area_m2"})
dfhi_ = _fc_to_df(stats_high).rename(columns={"sum":"highrise_area_m2"})

df_uc = dfp_.merge(dfh_, on=id_field, how="left")\
             .merge(dfb_, on=id_field, how="left")\
             .merge(dfhi_, on=id_field, how="left")
df_uc["built_area_km2"] = df_uc["built_area_m2"]/1e6
df_uc["highrise_share_pct"] = (df_uc["highrise_area_m2"]/df_uc["built_area_m2"]*100)\
    .replace([pd.NA, float("inf")], 0)

out_csv = f"exports/{OUT_PREFIX}_UC_stats.csv"
df_uc.to_csv(out_csv, index=False)
print("[OK] UC stats →", out_csv)

In [ ]:
# CSV (fast for Folium)
try:
    dfp = geemap.ee_to_df(pts_fc)
except Exception:
    info = ee.FeatureCollection(pts_fc).getInfo()
    dfp = pd.DataFrame([f["properties"] for f in info["features"]])
dfp = dfp.rename(columns={"latitude":"lat","longitude":"lon"})
dfp[["lat","lon","presence","height","highrise"]].to_csv(
    f"exports/{OUT_PREFIX}_points_{POINT_SAMPLE_M}m.csv", index=False)
print(f"[OK] CSV points → exports/{OUT_PREFIX}_points_{POINT_SAMPLE_M}m.csv")

# Shapefile set (.shp/.shx/.dbf/.prj)
gdf_pts = gpd.read_file(geojson_pts)
if gdf_pts.crs is None or gdf_pts.crs.to_epsg() != 4326:
    gdf_pts = gdf_pts.set_crs(4326, allow_override=True)
shp_dir = f"exports/{OUT_PREFIX}_points_{POINT_SAMPLE_M}m_shp"
os.makedirs(shp_dir, exist_ok=True)
gdf_pts.to_file(os.path.join(shp_dir, f"{OUT_PREFIX}_points_{POINT_SAMPLE_M}m.shp"), driver="ESRI Shapefile")
print(f"[OK] Shapefile set → {shp_dir}")

# ---------- 9) UC-wise statistics ----------
# Built area = sum(pixel_area where presence≥P), Height stats within built pixels, High-rise share, Fractional counts.
pixel_area = ee.Image.pixelArea()  # m²

built_area_m2_img = pixel_area.updateMask(built_mask)
highrise_area_m2_img = pixel_area.updateMask(highrise_mask)

reducers = ee.Reducer.mean().combine(
    reducer2=ee.Reducer.median(), sharedInputs=True
).combine(
    reducer2=ee.Reducer.minMax(), sharedInputs=True
)

# Presence: mean presence (0..1) as intensity proxy
stats_presence = presence.reduceRegions(ucs_fc, ee.Reducer.mean(), scale=4, tileScale=2)
# Height stats (only where built)
stats_height = height_built.reduceRegions(ucs_fc, reducers, scale=4, tileScale=2)
# Areas
stats_built_area = built_area_m2_img.reduceRegions(ucs_fc, ee.Reducer.sum(), scale=4, tileScale=2)
stats_highrise_area = highrise_area_m2_img.reduceRegions(ucs_fc, ee.Reducer.sum(), scale=4, tileScale=2)
# Fractional counts (sum over area; for counts follow EE example scripts)
stats_frac = frac_cnt.reduceRegions(ucs_fc, ee.Reducer.mean(), scale=4, tileScale=2)

# Convert to DataFrames (safe fallbacks)
def _fc_to_df(fc):
    try:
        return geemap.ee_to_df(fc)
    except Exception:
        info = ee.FeatureCollection(fc).getInfo()
        return pd.DataFrame([f["properties"] for f in info["features"]])

df_presence = _fc_to_df(stats_presence).rename(columns={"mean":"presence_mean"})
df_height   = _fc_to_df(stats_height).rename(columns={
    "mean":"height_mean","median":"height_median","min":"height_min","max":"height_max"})
df_barea    = _fc_to_df(stats_built_area).rename(columns={"sum":"built_area_m2"})
df_hrarea   = _fc_to_df(stats_highrise_area).rename(columns={"sum":"highrise_area_m2"})
df_frac     = _fc_to_df(stats_frac).rename(columns={"mean":"frac_count_mean"})

# Merge on UC id
dfs = [df_presence, df_height, df_barea, df_hrarea, df_frac]
df_uc = dfs[0]
for d in dfs[1:]:
    common = set(df_uc.columns).intersection(d.columns)
    keys = [id_field] if id_field in common else list(common)  # fall back to overlap
    df_uc = df_uc.merge(d, on=keys, how="left")

# Add coverage and shares
df_uc["built_area_km2"] = df_uc["built_area_m2"] / 1e6
df_uc["highrise_area_km2"] = df_uc["highrise_area_m2"] / 1e6
df_uc["highrise_share_pct"] = (df_uc["highrise_area_m2"] / df_uc["built_area_m2"] * 100).replace([pd.NA, float("inf")], 0)

out_csv = f"exports/{OUT_PREFIX}_UC_stats.csv"
df_uc.to_csv(out_csv, index=False)
print(f"[OK] UC stats → {out_csv}")

# ---------- 10) Quick preview map (optional, ipyleaflet) ----------
try:
    m = geemap.Map(center=[31.5204, 74.3587], zoom=12)
    m.addLayer(presence, vis_presence, f"Presence {YEAR}")
    m.addLayer(height_built, vis_height, f"Height {YEAR} (m)")
    m.addLayer(highrise_mask.selfMask(), {"min":0,"max":1,"palette":["yellow","red"]}, f"High-rise ≥{HIGHRISE_HEIGHT}m")
    m.addLayer(ucs_fc.style(color="black", fillColor="00000000", width=1), {}, "UCs")
    m.to_html(f"exports/{OUT_PREFIX}_preview.html")
    print(f"[OK] Saved HTML preview → exports/{OUT_PREFIX}_preview.html")
except Exception as e:
    print("[INFO] Preview skipped:", e)